# Session 7 lab: Derivatives

**Mathematics for AI and Machine Learning, KLAi Professional Learning**

Today you build a slope estimator from two nearby points (a finite difference), use it to check the derivative rules, and then use it on the loss of a tiny model. The data are four illustrative deliveries (made-up numbers): the distance x in km and the fee y in thousand UGX. The spreadsheet lab uses the same numbers.

Rhythm of every step: **PREDICT**, run, **READ**, **CHANGE one thing**. Run the cells in order from the top. A cell that says `answer = None  # your code here` is yours to fill. Lines that start with `check(` tell you whether your answer is right.

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 0: Set up

We use only numpy (numbers), pandas (tables) and matplotlib (charts). Run the cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("Ready. numpy", np.__version__)

---
## Step 1: The slope from two points

A slope is the change in output divided by the change in input. For a curve we use two points that are close together. The function is f(x) = x squared and the point is x = 3. Here the step is h = 1, so the second point is x = 4.

**PREDICT first.** For f(x) = x squared, what is the slope between x = 3 and x = 4? Work it out on paper: f(4) minus f(3), divided by 1.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def f(x):
    return x ** 2

x0 = 3
h = 1
# Hint: (f(x0 + h) - f(x0)) / h
slope_h1 = None  # your code here
print('slope with h = 1:', slope_h1)
check('slope of x^2 at 3 with h = 1', slope_h1, 7)

**READ the output.** Is the number the slope of the curve at x = 3, or the slope of the line joining two points? Look at the picture on the slides.

> My answer: 

---
## Step 2: Shrink the step h

The derivative is the value these slopes settle on as h gets smaller. Build the whole table at once: one slope for each h.

**PREDICT first.** As h gets smaller, what number do the slopes get close to? Guess before you run.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
hs = [2, 1, 0.5, 0.1, 0.01, 0.001]
# Hint: a list with (f(x0 + h) - f(x0)) / h for each h in hs
slopes = None  # your code here
table = pd.DataFrame({'h': hs, 'slope': slopes})
print(table)
check('slopes for the six values of h', slopes, [8, 7, 6.5, 6.1, 6.01, 6.001], tol=0.0005)

**READ the output.** What do the slopes approach? Which h gives the answer closest to it?

> My answer: 

**CHANGE one thing.** In a new cell, add h = 1e-12 to the list and run it again. What happened to the slope? Why would a computer have trouble here? (It subtracts two numbers that are almost equal.)

> What I saw: 

---
## Step 3: A better estimate: the central difference

Looking on both sides of x gives a more accurate slope. You write one function, slope, and use it for the rest of the lab. Then you test it on the rules from the slides.

**PREDICT first.** For x cubed at x = 2 the power rule says the slope is 3 times 2 squared. What number is that?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def slope(f, x, h=1e-5):
    # Hint: (f(x + h) - f(x - h)) / (2 * h)
    return None  # your code here

print('x^2 at 3:', slope(f, 3))

In [ ]:
# Each case is: name, function, point, exact slope from the rule
cases = [('x^3 at 2 (power rule)', lambda x: x ** 3, 2, 12.0),
         ('e^x at 1 (exponential rule)', np.exp, 1, np.e),
         ('ln x at 2 (logarithm rule, 1/x)', np.log, 2, 0.5),
         ('x^2 e^x at 1 (product rule, 3e)', lambda x: x ** 2 * np.exp(x), 1, 3 * np.e),
         ('x^2/(x+1) at 1 (quotient rule)', lambda x: x ** 2 / (x + 1), 1, 0.75)]
for name, g, x, exact in cases:
    estimate = slope(g, x)
    print(name, '| finite difference:', estimate, '| rule:', exact)
    check(name, estimate, exact, tol=0.0001)

**READ the output.** For each case, how many decimals does the finite difference share with the rule? If one case disagrees, which of the two would you check first?

> My answer: 

---
## Step 4: The loss of a one-parameter model

The model predicts the fee as w times the distance. The loss is the mean of the squared errors. Write it as a function of w so that you can try many values.

**PREDICT first.** At w = 1 the predictions are 1, 2, 3, 4 and the fees are 3, 5, 6, 9. The errors are -2, -3, -3, -5. What is the mean of their squares?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
x = np.array([1, 2, 3, 4])   # distance in km
y = np.array([3, 5, 6, 9])   # fee in thousand UGX
n_rows = len(x)

def loss(w):
    # Hint: the mean of (w * x - y) ** 2
    return None  # your code here

print('L(1) =', loss(1), '  L(2) =', loss(2), '  L(3) =', loss(3))
check('L(1)', loss(1), 11.75)
check('L(2)', loss(2), 0.75)

**READ the output.** Which of w = 1, 2 and 3 gives the smallest loss? Does that match the bowl on the slides?

> My answer: 

---
## Step 5: The derivative of the loss, checked by numbers

The slides give dL/dw = 2/n times the sum of x times (w x minus y). Write it, then compare it with your central difference.

**PREDICT first.** At w = 1 the slides say the slope is -18.5. At w = 3 it is +11.5. What does the sign tell you about the loss?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def dloss(w):
    # Hint: 2 / n_rows times the sum of x * (w * x - y)
    return None  # your code here

print('formula   :', dloss(1), dloss(3))
print('finite diff:', slope(loss, 1), slope(loss, 3))
check('slope of the loss at w = 1', dloss(1), -18.5)
check('slope of the loss at w = 3', dloss(3), 11.5)
check('formula agrees with the finite difference', [dloss(1), dloss(3)], [slope(loss, 1), slope(loss, 3)], tol=0.0001)

**READ the output.** The two printed lines should agree to many decimals. If they do not, which one would you trust less, the formula or the finite difference, and how would you find out?

> My answer: 

---
## Step 6: Which way to move w

Move w opposite to the sign of the slope: a negative slope means raise w, a positive slope means lower w. Turn that rule into a number: +1 for raise, -1 for lower.

**PREDICT first.** At w = 2 the slope of the loss is -3.5. Should w go up or down? Is the loss near its bottom there?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def move(w):
    # Hint: +1 if dloss(w) is negative (raise w), otherwise -1 (lower w)
    return None  # your code here

for w in [0, 1, 2, 3, 4]:
    print('w =', w, '| loss =', round(float(loss(w)), 3), '| slope =', round(float(dloss(w)), 3), '| move:', move(w))
check('direction at w = 1 and at w = 3', [move(1), move(3)], [1, -1])

**READ the output.** Where does the slope change sign? Which two values of w is the bottom between?

> My answer: 

---
## Step 7: The bottom of the bowl

At the lowest point the slope is zero. Solving dL/dw = 0 gives w = (sum of x times y) divided by (sum of x squared). Then draw the loss curve with the tangent line at w = 1.

**PREDICT first.** The sum of x times y is 67 and the sum of x squared is 30. What is w?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: the sum of x * y divided by the sum of x * x
w_star = None  # your code here
print('best w:', w_star)
check('best w', w_star, 2.2333, tol=0.001)
check('loss at the best w', loss(w_star), 0.3417, tol=0.001)
check('slope at the best w is zero', dloss(w_star), 0, tol=0.0001)

In [ ]:
ws = np.linspace(0, 4, 200)
plt.figure(figsize=(6.4, 3.8))
plt.plot(ws, [loss(w) for w in ws], color='black', linewidth=2.5, label='loss L(w)')
tangent = loss(1) + dloss(1) * (ws - 1)
plt.plot(ws, tangent, color='#AE2525', linewidth=2, label='tangent at w = 1 (slope -18.5)')
plt.scatter([1, 2.2333], [loss(1), loss(2.2333)], color='#E0BE36', edgecolor='black', zorder=5, s=70)
plt.ylim(0, 40)
plt.xlabel('parameter w')
plt.ylabel('loss L(w)')
plt.legend()
plt.show()

**CHANGE one thing.** In a new cell, change the last fee from 9 to 10, recompute w_star and the loss. Does the best w go up or down? Predict first. (Answer: w goes up to 2.367, because a bigger fee needs a bigger fee per km.)

> What I saw: 

---
## Finished

Run the next cell for your automatic check. Then use File, Save. Keep three numbers: -18.5, +11.5 and 2.233.

In [ ]:
show_checks()